In [1]:
!pip install -qU \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-huggingface \
transformers \
sentence-transformers \
faiss-cpu \
pypdf \
accelerate

In [2]:
from google.colab import files
uploaded = files.upload()

Saving sample.pdf to sample (1).pdf


In [3]:
!pip install pymupdf

In [44]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_path = "sample.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

print("Pages loaded:", len(documents))
print(documents[0].page_content[:500])

Pages loaded: 10
ABSTRACT 
With the proliferation of advanced communication technologies and the deepening 
interdependence between cyber and physical components, power distribution networks are 
subject to miscellaneous security risks induced by malicious attackers. To address the issue, 
this paper proposes a security risk assessment method and a risk-oriented defense resource 
allocation strategy for cyber-physical distribution networks (CPDNs) against coordinated 
cyber-attacks. First, an attack graph-based 


In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

docs = text_splitter.split_documents(documents)

print("Chunks created:", len(docs))
print(docs[0].page_content[:300])

Chunks created: 33
ABSTRACT 
With the proliferation of advanced communication technologies and the deepening 
interdependence between cyber and physical components, power distribution networks are 
subject to miscellaneous security risks induced by malicious attackers. To address the issue, 
this paper proposes a secu


In [78]:
import re

def clean_text(text):
    text = re.sub(r'\$.*?\$', '', text)
    text = re.sub(r'\\[a-zA-Z]+\{.*?\}', '', text)
    text = re.sub(r'[_^{}\\]', ' ', text)
    text = re.sub(r'(\w)\n(\w)', r'\1 \2', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)
docs = text_splitter.split_documents(documents)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def ask_rag(question):
    retrieved_docs = retriever.invoke(question)
    context = " ".join([d.page_content for d in retrieved_docs])
    prompt = f"Answer the question based on the context.\nContext: {context[:1200]}\nQuestion: {question}\nAnswer:"
    return generate_answer(prompt)

In [79]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [80]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

print("Vector DB created")

Vector DB created


In [81]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [62]:
!pip install groq

In [82]:
from groq import Groq

client = Groq(api_key="YOUR_GROQ_API_KEY_HERE")

def ask_rag(question):
    retrieved_docs = retriever.invoke(question)
    context = " ".join([d.page_content for d in retrieved_docs])

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer questions using only the provided context."
            },
            {
                "role": "user",
                "content": f"Context:\n{context[:3000]}\n\nQuestion: {question}\n\nAnswer:"
            }
        ]
    )
    return response.choices[0].message.content

In [83]:
query = "What is this document about?"
docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---\n")
    print(doc.page_content[:400])


--- Chunk 1 ---

. It supports secure user authentication and ensures that only authorized personnel can access the control environment. Users can upload sensitive operational files, which are automatically encrypted to maintain confidentiality during transmission and storage. The MM Interface also manages the assignment of these encrypted files to available routers or servers within the CPDN

--- Chunk 2 ---

. The module allows authorized downloading of files, enabling seamless data exchange across the network. It is also capable of logging access activities, making it easier to trace malicious interactions or attempted intrusions.

--- Chunk 3 ---

. It also processes storage assignment requests, ensuring that uploaded file data is routed to the correct devices for operational use. Through this centralized management interface, administrators maintain oversight of the entire network infrastructure, ensuring that both cyber and physical components function harmoniously and securely


In [65]:
!pip install gradio

In [84]:
print("ask_rag exists:", "ask_rag" in globals())

ask_rag exists: True


In [85]:
import gradio as gr

def chat_fn(message, history):
    return ask_rag(message)

demo = gr.ChatInterface(fn=chat_fn)
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://97a31d15a746d7f3d8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
